# Day06下午个人项目：电商用户数据可视化

姓名/学号或GitHub用户名：王仁献24012477 
第5天专题（A/B/C/D/E）：A

本Notebook需要完成4张独立图、1张综合图和1份图表清单。请阅读`docs/day06_student_visualization_manual.md`后开始。


## 项目规则

1. 使用第4天清洗数据，并核对第5天个人分析结果；
2. 柱状图和散点图必做；折线图只能用于时间或有序阶段；
3. 饼图只用于少量类别的整体构成，必要时改用柱状图；
4. 每张图写“观察—证据—边界”；
5. 输出文件名和目录不得修改，以便第7天Flask直接复用。


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

STUDENT_ID = "王仁献" \
"24012477"
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "PingFang SC",
    "Heiti SC", "Arial Unicode MS", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到第4天清洗数据，请先完成Day04。")


ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
DAY05_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR = ROOT / "output" / "day06_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("学生：", STUDENT_ID)
print("专题：", TOPIC)
print("输出：", OUTPUT_DIR.relative_to(ROOT))


学生： 王仁献24012477
专题： A
输出： output\day06_visualization


## 检查点1：输入与业务问题

先验证4个输入文件，再写出4个问题。不要在尚未理解指标时直接绘图。


In [2]:
required_inputs = [
    DATA_PATH,
    DAY05_DIR / "overall_metrics.csv",
    DAY05_DIR / "segment_analysis.csv",
    DAY05_DIR / "cross_analysis.csv",
]
missing_inputs = [str(path.relative_to(ROOT)) for path in required_inputs if not path.exists()]
assert not missing_inputs, f"缺少输入文件：{missing_inputs}"

df = pd.read_csv(DATA_PATH)
overall_metrics = pd.read_csv(required_inputs[1])
segment_analysis = pd.read_csv(required_inputs[2])
cross_analysis = pd.read_csv(required_inputs[3])

assert df.shape[0] == 5630, f"清洗数据行数异常：{df.shape}"
assert {"CustomerID", "Churn", "TenureGroup", "OrderCount", "CashbackAmount"}.issubset(df.columns)
assert set(df["Churn"].dropna().unique()).issubset({0, 1})

display(overall_metrics)
display(segment_analysis.head())
display(cross_analysis.head())
print("检查点1A通过：输入文件有效")


,指标,值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


,TenureGroup,用户数,流失人数,流失率,平均订单数,订单数中位数,平均优惠券数,平均返现,平均App使用时长,平均满意度,平均距上次下单天数
0,0-3个月,1560,653,0.42,2.32,2.00,1.46,155.66,2.98,3.18,3.47
1,12个月以上,1896,95,0.05,3.67,2.00,2.00,208.86,2.92,3.08,5.30
2,4-6个月,590,44,0.07,2.97,2.00,1.80,169.91,3.02,2.94,4.57
3,7-12个月,1584,156,0.10,2.75,2.00,1.60,163.31,2.88,2.99,4.38


,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
0,12个月以上,0,1357,43,0.03,3.82,可观察
1,7-12个月,0,1178,75,0.06,2.78,可观察
2,0-3个月,0,1038,308,0.30,2.20,可观察
3,12个月以上,1,539,52,0.10,3.28,可观察
4,0-3个月,1,522,345,0.66,2.55,可观察


检查点1A通过：输入文件有效


In [3]:
# TODO：填写4个业务问题和图表选择理由
business_questions = {
    "category_bar": "不同等级用户的流失率是否存在差异？",
    "behavior_scatter": "OrderCount与CashbackAmount呈现什么关系？",
    "ordered_line": "用户流失率如何随TenureGroup阶段变化？",
    "composition_chart": "全部用户由哪些会员等级类别构成？",
}

chart_reasons = {
    "category_bar": "柱状图适合比较不同离散类别（例如会员等级）之间的用户数和流失率差异。",
    "behavior_scatter": "散点图适合展示两个连续变量之间的分布与关系，且可通过颜色区分Churn状态。",
    "ordered_line": "折线图适合展示按TenureGroup排序的阶段性变化趋势，体现流失率随阶段变化的走势。",
    "composition_chart": "构成图适合展示用户在少量类别（如会员等级）上的整体比例结构，便于观察主要类别份额。",
}

assert all(text.strip() for text in business_questions.values()), "请填写4个业务问题"
assert all(text.strip() for text in chart_reasons.values()), "请填写4个图表选择理由"
print("检查点1B通过：业务问题和选择理由已填写")


检查点1B通过：业务问题和选择理由已填写


## 任务1：类别比较柱状图

要求：选择一个离散分组字段，计算用户数和一个核心指标；若绘制比率，标签中必须同时给出样本量。


In [4]:
# TODO：完成绘图数据。建议使用自己的第5天主分组字段。
category_field = "TenureGroup"
category_summary = (
    df.groupby(category_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)

assert category_field in df.columns, "category_field必须是有效字段"
assert isinstance(category_summary, pd.DataFrame), "category_summary必须是DataFrame"
assert {category_field, "用户数"}.issubset(category_summary.columns)
display(category_summary)


,TenureGroup,用户数,流失率
0,0-3个月,1560,0.42
1,12个月以上,1896,0.05
2,4-6个月,590,0.07
3,7-12个月,1584,0.10


In [5]:
# TODO：绘制并保存柱状图
fig_bar, ax_bar = plt.subplots(figsize=(10, 6))

category_summary = category_summary.sort_values(by=category_field)
ax_bar.bar(category_summary[category_field], category_summary["流失率"], color="#4C72B0")
for idx, row in category_summary.iterrows():
    ax_bar.text(
        idx,
        row["流失率"] + 0.01,
        f"{row['流失率']:.1%}\n(n={row['用户数']})",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax_bar.set_title("不同TenureGroup用户的流失率比较", fontsize=14, fontweight="bold")
ax_bar.set_xlabel("TenureGroup")
ax_bar.set_ylabel("流失率")
ax_bar.yaxis.set_major_formatter(PercentFormatter(1.0))
ax_bar.set_ylim(0, category_summary["流失率"].max() * 1.2)
ax_bar.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=15)

bar_path = OUTPUT_DIR / "01_category_bar.png"
fig_bar.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.close(fig_bar)

assert bar_path.exists() and bar_path.stat().st_size > 0, "柱状图尚未保存"
print("已输出：", bar_path.relative_to(ROOT))


已输出： output\day06_visualization\01_category_bar.png


### 柱状图结论

- 观察：不同TenureGroup用户的流失率存在明显差异，新用户（0-3个月）流失率显著高于其他阶段。
- 证据：0-3个月组流失率约为41.9%，而12个月以上组流失率仅约5.0%；样本量分别为1560和1896。
- 边界：该图仅显示不同阶段的流失率差异，不能说明流失原因，也不能直接推断阶段间因果关系。

## 任务2：用户行为散点图

要求：选择两个数值字段，一行代表一个用户，颜色区分`Churn`，设置透明度。


In [6]:
# TODO：选择两个数值字段，例如OrderCount与CashbackAmount
x_field = "OrderCount"
y_field = "CashbackAmount"

assert x_field in df.columns and y_field in df.columns
assert pd.api.types.is_numeric_dtype(df[x_field])
assert pd.api.types.is_numeric_dtype(df[y_field])

fig_scatter, ax_scatter = plt.subplots(figsize=(10, 6))

for churn_value, color, label in [(0, "#4C72B0", "留存用户"), (1, "#DD8452", "流失用户")]:
    subset = df[df["Churn"] == churn_value]
    ax_scatter.scatter(
        subset[x_field],
        subset[y_field],
        s=20,
        alpha=0.6,
        c=color,
        label=label,
        edgecolors="none",
    )

ax_scatter.set_title("OrderCount与CashbackAmount的用户行为关系", fontsize=14, fontweight="bold")
ax_scatter.set_xlabel("OrderCount")
ax_scatter.set_ylabel("CashbackAmount")
ax_scatter.legend(title="Churn")
ax_scatter.grid(True, linestyle="--", alpha=0.3)

scatter_path = OUTPUT_DIR / "02_behavior_scatter.png"
fig_scatter.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.close(fig_scatter)

assert scatter_path.exists() and scatter_path.stat().st_size > 0, "散点图尚未保存"
print("已输出：", scatter_path.relative_to(ROOT))


已输出： output\day06_visualization\02_behavior_scatter.png


### 散点图结论

- 观察：留存用户与流失用户在OrderCount和CashbackAmount上的分布有差异，流失用户更集中在低订单数和低返现金额区域。
- 证据：留存用户点更分散、现金返还金额和订单数普遍较高，而流失用户多数集中在OrderCount较低、CashbackAmount较低的区域。
- 边界：该图只反映变量间的关联分布，不能证明OrderCount或CashbackAmount是导致流失的直接原因。

## 任务3：有序阶段折线图

当前数据没有日期。建议使用`TenureGroup`或`SatisfactionScore`，并明确写成“阶段比较”。


In [7]:
TENURE_ORDER = ["0-3个月", "4-6个月", "7-12个月", "12个月以上"]

# TODO：准备有序绘图数据
ordered_field = "TenureGroup"
ordered_summary = (
    df.groupby(ordered_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)
ordered_summary[ordered_field] = pd.Categorical(
    ordered_summary[ordered_field], categories=TENURE_ORDER, ordered=True
)
ordered_summary = ordered_summary.sort_values(by=ordered_field)

assert ordered_field in {"TenureGroup", "SatisfactionScore"}, \
    "本项目折线图只允许使用具有明确顺序的TenureGroup或SatisfactionScore"
assert isinstance(ordered_summary, pd.DataFrame)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)
display(ordered_summary)

,TenureGroup,用户数,流失率
0,0-3个月,1560,0.42
2,4-6个月,590,0.07
3,7-12个月,1584,0.10
1,12个月以上,1896,0.05


In [8]:
# TODO：绘制折线图；若绘制流失率，应标注比例和样本量
fig_line, ax_line = plt.subplots(figsize=(10, 6))

ax_line.plot(
    ordered_summary[ordered_field].astype(str),
    ordered_summary["流失率"],
    marker="o",
    color="#4C72B0",
    linewidth=2,
)
for idx, row in ordered_summary.iterrows():
    ax_line.text(
        idx,
        row["流失率"] + 0.01,
        f"{row['流失率']:.1%}\n(n={row['用户数']})",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax_line.set_title("不同TenureGroup阶段的流失率变化", fontsize=14, fontweight="bold")
ax_line.set_xlabel("TenureGroup阶段")
ax_line.set_ylabel("流失率")
ax_line.yaxis.set_major_formatter(PercentFormatter(1.0))
ax_line.set_ylim(0, ordered_summary["流失率"].max() * 1.3)
ax_line.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=15)

line_path = OUTPUT_DIR / "03_ordered_line.png"
fig_line.savefig(line_path, dpi=150, bbox_inches="tight")
plt.close(fig_line)

assert line_path.exists() and line_path.stat().st_size > 0, "折线图尚未保存"
print("已输出：", line_path.relative_to(ROOT))

已输出： output\day06_visualization\03_ordered_line.png


### 折线图结论

- 观察：不同TenureGroup阶段的流失率总体呈下降趋势，0-3个月阶段最高，12个月以上阶段最低。
- 证据：0-3个月组流失率约41.9%，4-6个月组约7.5%，7-12个月组约9.8%，12个月以上组约5.0%；样本量分别为1560、590、1584、1896。
- 边界：该图反映阶段性流失率变化，不代表时间序列的月度或年度趋势，也不说明阶段间的因果关系。

## 任务4：整体构成图

类别少于或等于5个时可以使用饼图或环形图；否则改用柱状图。必须在选择理由中说明判断。


In [9]:
# TODO：选择构成字段并准备汇总表
composition_field = "PreferredPaymentMode"
composition_summary = (
    df.groupby(composition_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"))
      .reset_index()
)
composition_summary["占比"] = composition_summary["用户数"] / composition_summary["用户数"].sum()
composition_summary = composition_summary.sort_values(by="占比", ascending=False)

assert composition_field in df.columns
assert isinstance(composition_summary, pd.DataFrame)
assert {composition_field, "用户数", "占比"}.issubset(composition_summary.columns)
assert np.isclose(composition_summary["占比"].sum(), 1.0), "构成占比之和应为1"
display(composition_summary)

,PreferredPaymentMode,用户数,占比
2,Debit Card,2314,0.41
1,Credit Card,1774,0.32
3,E wallet,614,0.11
0,Cash on Delivery,514,0.09
4,UPI,414,0.07


In [10]:
# TODO：类别不超过5个时绘制环形图，否则改用柱状图
fig_composition, ax_composition = plt.subplots(figsize=(10, 6))
labels = composition_summary[composition_field].tolist()
sizes = composition_summary["占比"].tolist()
colors = plt.cm.Pastel1(np.linspace(0, 1, len(labels)))

wedges, texts, autotexts = ax_composition.pie(
    sizes,
    labels=labels,
    autopct="%1.1f%%",
    startangle=140,
    colors=colors,
    wedgeprops=dict(width=0.4, edgecolor="w"),
    textprops={"fontsize": 10},
)

ax_composition.set_title("用户首选支付方式构成", fontsize=14, fontweight="bold")
ax_composition.axis("equal")

composition_path = OUTPUT_DIR / "04_composition_chart.png"
fig_composition.savefig(composition_path, dpi=150, bbox_inches="tight")
plt.close(fig_composition)

assert composition_path.exists() and composition_path.stat().st_size > 0, "构成图尚未保存"
print("已输出：", composition_path.relative_to(ROOT))

已输出： output\day06_visualization\04_composition_chart.png


### 构成图结论

- 观察：用户首选支付方式主要集中在Debit Card和Credit Card，两者合计占比超过60%。
- 证据：Debit Card约占41.1%，Credit Card约占31.5%，其余三种方式合计约27.4%。
- 边界：该图展示的是支付方式构成，不适合用于判断不同支付方式对流失率或用户价值的因果影响。

## 检查点2与3：基础图表、优化和解释

逐项使用`docs/day06_chart_checklist.md`检查。确认比率图给出样本量、中文正常、颜色含义一致。


In [11]:
individual_paths = [bar_path, scatter_path, line_path, composition_path]
for path in individual_paths:
    assert path.exists() and path.suffix.lower() == ".png"
    assert path.stat().st_size > 5_000, f"图片可能为空或质量过低：{path.name}"

print("检查点2通过：4张独立图已生成")
print("检查点3需要结合图表和文字结论人工复核")


检查点2通过：4张独立图已生成
检查点3需要结合图表和文字结论人工复核


## 任务5：2×2综合图

重新在4个子图中绘制核心内容，不要把4张PNG作为截图拼接。统一标题、颜色、字体和留白。


In [12]:
fig_summary, axes = plt.subplots(2, 2, figsize=(14, 10))

# 子图1：TenureGroup流失率对比
ax = axes[0, 0]
category_summary = category_summary.sort_values(by=category_field)
ax.bar(category_summary[category_field], category_summary["流失率"], color="#4C72B0")
for idx, row in category_summary.iterrows():
    ax.text(idx, row["流失率"] + 0.01, f"{row['流失率']:.1%}", ha="center", va="bottom", fontsize=9)
ax.set_title("TenureGroup流失率比较")
ax.set_xlabel("TenureGroup")
ax.set_ylabel("流失率")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.tick_params(axis='x', rotation=15)

# 子图2：OrderCount vs CashbackAmount散点
ax = axes[0, 1]
for churn_value, color, label in [(0, "#4C72B0", "留存用户"), (1, "#DD8452", "流失用户")]:
    subset = df[df["Churn"] == churn_value]
    ax.scatter(subset["OrderCount"], subset["CashbackAmount"], s=20, alpha=0.6, c=color, label=label, edgecolors="none")
ax.set_title("OrderCount与CashbackAmount关系")
ax.set_xlabel("OrderCount")
ax.set_ylabel("CashbackAmount")
ax.legend(title="Churn")
ax.grid(True, linestyle="--", alpha=0.3)

# 子图3：TenureGroup阶段流失率趋势
ax = axes[1, 0]
ax.plot(ordered_summary[ordered_field].astype(str), ordered_summary["流失率"], marker="o", color="#4C72B0", linewidth=2)
for idx, row in ordered_summary.iterrows():
    ax.text(idx, row["流失率"] + 0.01, f"{row['流失率']:.1%}", ha="center", va="bottom", fontsize=9)
ax.set_title("TenureGroup阶段流失率变化")
ax.set_xlabel("TenureGroup阶段")
ax.set_ylabel("流失率")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.tick_params(axis='x', rotation=15)

# 子图4：支付方式构成
ax = axes[1, 1]
labels = composition_summary[composition_field].tolist()
sizes = composition_summary["占比"].tolist()
colors = plt.cm.Pastel1(np.linspace(0, 1, len(labels)))
ax.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=140, colors=colors, wedgeprops=dict(width=0.4, edgecolor="w"), textprops={"fontsize": 9})
ax.set_title("支付方式构成")
ax.axis("equal")

fig_summary.suptitle("电商用户数据可视化分析概览", fontsize=16, fontweight="bold")
fig_summary.tight_layout(rect=[0, 0, 1, 0.96])
fig_summary.subplots_adjust(hspace=0.35, wspace=0.3)

summary_path = OUTPUT_DIR / "day06_visualization_summary.png"
fig_summary.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.close(fig_summary)

assert summary_path.exists() and summary_path.stat().st_size > 0, "综合图尚未保存"
print("已输出：", summary_path.relative_to(ROOT))


已输出： output\day06_visualization\day06_visualization_summary.png


## 综合发现与局限

1. 综合发现1：新用户（0-3个月）流失率远高于其他阶段，说明用户初期体验或启动阶段对留存影响最大。证据见10类别比较柱状图，0-3个月组流失率达41.9%。
2. 综合发现2：OrderCount与CashbackAmount的散点图显示，流失用户更倾向于低订单数和低返现金额，提示活跃度和返现激励之间存在关联。证据见散点图中流失用户点主要集中在低值区域。
3. 综合发现3：有序阶段折线图显示流失率随TenureGroup阶段总体下降，成熟用户（12个月以上）流失率最低，说明用户留存稳定性随使用时长提升。证据见折线图中的阶段趋势。
4. 数据或方法局限：本分析基于历史清洗数据，仅观察变量关联，无法确认因果关系；同时未考虑订单频率、复购间隔等其他潜在影响因素。

注意：`CashbackAmount`是返现金额，不是销售额、收入或GMV。


## 任务6：图表清单与检查点4

清单是第7天Flask读取图表说明的基础。每张图填写业务问题、图表类型、主要发现和局限。


In [13]:
# TODO：填写5行清单，不得保留“请填写”
chart_manifest = pd.DataFrame([
    {"chart_id": "01", "file_name": "01_category_bar.png", "business_question": "不同用户生命周期阶段的流失率如何分布？", "chart_type": "bar", "key_finding": "0-3个月新用户流失率显著最高，超过40%，表明初期留存是关键。", "limitation": "未考虑阶段内用户人口统计特征和事件触发因素。"},
    {"chart_id": "02", "file_name": "02_behavior_scatter.png", "business_question": "订单次数与返现金额是否与用户流失相关？", "chart_type": "scatter", "key_finding": "低订单数和低返现用户更易流失，说明活跃度与激励水平可能有关。", "limitation": "散点图仅展示关联，不区分因果或其他影响因素。"},
    {"chart_id": "03", "file_name": "03_ordered_line.png", "business_question": "随用户使用时长变化，流失率是否呈阶段性趋势？", "chart_type": "line", "key_finding": "流失率随TenureGroup阶段整体下降，成熟用户留存更稳定。", "limitation": "忽略了组内订单频率、复购间隔等行为差异。"},
    {"chart_id": "04", "file_name": "04_composition_chart.png", "business_question": "各支付方式在用户群中的占比如何？", "chart_type": "pie_or_bar", "key_finding": "信用卡与借记卡占比最高，移动支付次之，显示支付偏好集中。", "limitation": "仅展示构成比例，不反映支付方式与流失或价值之间的关系。"},
    {"chart_id": "05", "file_name": "day06_visualization_summary.png", "business_question": "整体可视化分析能否揭示关键留存与行为模式？", "chart_type": "dashboard", "key_finding": "综合看：初期流失高、低活跃用户易流失、长期用户留存率更高。", "limitation": "仪表盘基于有限变量，无法代表所有客户行为和因果机制。"},
])

assert len(chart_manifest) == 5
assert not chart_manifest.astype(str).apply(lambda col: col.str.contains("请填写").any()).any(), \
    "请完成图表清单"

manifest_path = OUTPUT_DIR / "chart_manifest.csv"
chart_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(chart_manifest)


,chart_id,file_name,business_question,chart_type,key_finding,limitation
0,01,01_category_bar.png,不同用户生命周期阶段的流失率如何分布？,bar,0-3个月新用户流失率显著最高，超过40%，表明初期留存是关键。,未考虑阶段内用户人口统计特征和事件触发因素。
1,02,02_behavior_scatter.png,订单次数与返现金额是否与用户流失相关？,scatter,低订单数和低返现用户更易流失，说明活跃度与激励水平可能有关。,散点图仅展示关联，不区分因果或其他影响因素。
2,03,03_ordered_line.png,随用户使用时长变化，流失率是否呈阶段性趋势？,line,流失率随TenureGroup阶段整体下降，成熟用户留存更稳定。,忽略了组内订单频率、复购间隔等行为差异。
3,04,04_composition_chart.png,各支付方式在用户群中的占比如何？,pie_or_bar,信用卡与借记卡占比最高，移动支付次之，显示支付偏好集中。,仅展示构成比例，不反映支付方式与流失或价值之间的关系。
4,05,day06_visualization_summary.png,整体可视化分析能否揭示关键留存与行为模式？,dashboard,综合看：初期流失高、低活跃用户易流失、长期用户留存率更高。,仪表盘基于有限变量，无法代表所有客户行为和因果机制。


In [14]:
required_outputs = [
    OUTPUT_DIR / "01_category_bar.png",
    OUTPUT_DIR / "02_behavior_scatter.png",
    OUTPUT_DIR / "03_ordered_line.png",
    OUTPUT_DIR / "04_composition_chart.png",
    OUTPUT_DIR / "day06_visualization_summary.png",
    OUTPUT_DIR / "chart_manifest.csv",
]
missing_outputs = [str(path.relative_to(ROOT)) for path in required_outputs if not path.exists()]
assert not missing_outputs, f"缺少成果文件：{missing_outputs}"

manifest_check = pd.read_csv(OUTPUT_DIR / "chart_manifest.csv")
assert list(manifest_check.columns) == [
    "chart_id", "file_name", "business_question",
    "chart_type", "key_finding", "limitation",
]
assert set(manifest_check["file_name"]) == {path.name for path in required_outputs[:-1]}

print("检查点4通过：第6天成果物完整")
print("下一步：重启内核并从头运行，然后执行提交检查脚本并推送GitHub。")


检查点4通过：第6天成果物完整
下一步：重启内核并从头运行，然后执行提交检查脚本并推送GitHub。
